# 5-Year Pipeline: Part 5.2 - Global Model Inference & Evaluation
This notebook evaluates the Global Models (Base and Refined) and logs the results.

In [1]:
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
import json
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

def update_model_log(model_name, metrics, features_count, removed_features=0):
    log_file = 'models/model_performance_log.json'
    os.makedirs('models', exist_ok=True)
    log_data = {}
    if os.path.exists(log_file):
        with open(log_file, 'r') as f:
            try: log_data = json.load(f)
            except: pass
    log_data[model_name] = {
        "date_trained": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "metrics": metrics,
        "features_count": features_count,
        "removed_features": removed_features
    }
    with open(log_file, 'w') as f:
        json.dump(log_data, f, indent=4)
    print(f"✅ Updated JSON log for: {model_name}")

print("Loading Test Data...")
df = pd.read_csv('5_year_final_data.csv')
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df['date'] = pd.to_datetime(df['date'])

df_clean = df.dropna(subset=['max_gain_7d', 'max_loss_7d']).copy()
df_clean = df_clean.sort_values('date').reset_index(drop=True)

n = len(df_clean)
test_df = df_clean.iloc[int(n * 0.80):].copy()

with open('global_base_features.txt', 'r') as f: base_features = [line.strip() for line in f.readlines()]
with open('global_features.txt', 'r') as f: refined_features = [line.strip() for line in f.readlines()]

Loading Test Data...


In [2]:
print("\n--- EVALUATING BASE MODELS ---")
gain_base = joblib.load('models/global_gain_base.pkl')
drop_base = joblib.load('models/global_drop_base.pkl')
base_scaler = joblib.load('models/global_base_scaler.pkl')
base_imputer = joblib.load('models/global_base_imputer.pkl')

X_test_base = test_df[base_features]
X_test_s_base = base_scaler.transform(base_imputer.transform(X_test_base))

preds_gain_base = gain_base.predict(X_test_s_base)
preds_drop_base = drop_base.predict(X_test_s_base)

mae_gain_base = mean_absolute_error(test_df['max_gain_7d'], preds_gain_base)
mae_drop_base = mean_absolute_error(test_df['max_loss_7d'], preds_drop_base)

print(f"Base Gain Model MAE: {mae_gain_base:.2f}%")
print(f"Base Drop Model MAE: {mae_drop_base:.2f}%")

update_model_log("Global_Gain_Base", {"MAE": round(mae_gain_base,3)}, len(base_features), 0)
update_model_log("Global_Drop_Base", {"MAE": round(mae_drop_base,3)}, len(base_features), 0)


--- EVALUATING BASE MODELS ---
Base Gain Model MAE: 15.58%
Base Drop Model MAE: 9.77%
✅ Updated JSON log for: Global_Gain_Base
✅ Updated JSON log for: Global_Drop_Base


In [3]:
print("\n--- EVALUATING REFINED MODELS ---")
gain_ref = joblib.load('models/5_year_global_gain_model.pkl')
drop_ref = joblib.load('models/5_year_global_drop_model.pkl')
ref_scaler = joblib.load('models/5_year_global_scaler.pkl')
ref_imputer = joblib.load('models/5_year_global_imputer.pkl')

X_test_ref = test_df[refined_features]
X_test_s_ref = ref_scaler.transform(ref_imputer.transform(X_test_ref))

preds_gain_ref = gain_ref.predict(X_test_s_ref)
preds_drop_ref = drop_ref.predict(X_test_s_ref)

mae_gain_ref = mean_absolute_error(test_df['max_gain_7d'], preds_gain_ref)
mae_drop_ref = mean_absolute_error(test_df['max_loss_7d'], preds_drop_ref)

actual_up = test_df['max_gain_7d'] > test_df['max_loss_7d'].abs()
pred_up = pd.Series(preds_gain_ref) > pd.Series(preds_drop_ref).abs()
dir_acc = (actual_up.reset_index(drop=True) == pred_up.reset_index(drop=True)).mean() * 100

print(f"Refined Gain Model MAE: {mae_gain_ref:.2f}%")
print(f"Refined Drop Model MAE: {mae_drop_ref:.2f}%")
print(f"Directional Accuracy: {dir_acc:.1f}%")

update_model_log("Global_Gain_Refined", {"MAE": round(mae_gain_ref,3), "Directional_Accuracy": f"{dir_acc:.1f}%"}, len(refined_features), len(base_features) - len(refined_features))
update_model_log("Global_Drop_Refined", {"MAE": round(mae_drop_ref,3)}, len(refined_features), len(base_features) - len(refined_features))


--- EVALUATING REFINED MODELS ---


Refined Gain Model MAE: 15.93%
Refined Drop Model MAE: 9.91%
Directional Accuracy: 48.8%
✅ Updated JSON log for: Global_Gain_Refined
✅ Updated JSON log for: Global_Drop_Refined


In [4]:
print("\n--- EVALUATING INDIVIDUAL GLOBAL MODELS ---")
individual_dir = 'models/individual_model/global'
individual_logs = {}
mae_gain_list = []
mae_drop_list = []
dir_acc_list = []

for sym in test_df['symbol'].unique():
    gain_model_path = f"{individual_dir}/{sym}_gain_model.pkl"
    drop_model_path = f"{individual_dir}/{sym}_drop_model.pkl"
    if os.path.exists(gain_model_path) and os.path.exists(drop_model_path):
        sym_df = test_df[test_df['symbol'] == sym]
        if len(sym_df) == 0: continue
        
        try:
            gain_model = joblib.load(gain_model_path)
            drop_model = joblib.load(drop_model_path)
            scaler = joblib.load(f"{individual_dir}/{sym}_scaler.pkl")
            imputer = joblib.load(f"{individual_dir}/{sym}_imputer.pkl")
            
            X_test_sym = sym_df[refined_features]
            X_test_s = scaler.transform(imputer.transform(X_test_sym))
            
            preds_gain = gain_model.predict(X_test_s)
            preds_drop = drop_model.predict(X_test_s)
            
            mae_gain = mean_absolute_error(sym_df['max_gain_7d'], preds_gain)
            mae_drop = mean_absolute_error(sym_df['max_loss_7d'], preds_drop)
            
            actual_up = sym_df['max_gain_7d'] > sym_df['max_loss_7d'].abs()
            pred_up = pd.Series(preds_gain) > pd.Series(preds_drop).abs()
            dir_acc = (actual_up.reset_index(drop=True) == pred_up.reset_index(drop=True)).mean() * 100
            
            mae_gain_list.append(mae_gain)
            mae_drop_list.append(mae_drop)
            dir_acc_list.append(dir_acc)
            
            individual_logs[sym] = {
                "MAE_Gain": round(mae_gain, 3), 
                "MAE_Drop": round(mae_drop, 3),
                "Directional_Accuracy": f"{dir_acc:.1f}%"
            }
        except Exception as e:
            pass

if mae_gain_list:
    avg_mae_gain = np.mean(mae_gain_list)
    avg_mae_drop = np.mean(mae_drop_list)
    avg_dir_acc = np.mean(dir_acc_list)
    
    print(f"Individual Global Models (Avg): MAE Gain {avg_mae_gain:.2f}% | MAE Drop {avg_mae_drop:.2f}% | Dir Acc {avg_dir_acc:.1f}%")
    update_model_log("Individual_Global_Avg", {
        "MAE_Gain": round(avg_mae_gain,3),
        "MAE_Drop": round(avg_mae_drop,3),
        "Directional_Accuracy": f"{avg_dir_acc:.1f}%"
    }, len(refined_features))
    
    with open('models/individual_global_log.json', 'w') as f:
        json.dump(individual_logs, f, indent=4)
    print("✅ Saved individual global logs to models/individual_global_log.json")




--- EVALUATING INDIVIDUAL GLOBAL MODELS ---
